# Experiment Report Plots

Run this notebook from the `main/experiments` folder, or from the project root. It scans experiment subfolders that contain `history.csv`, creates one training curve plot per experiment, and writes a consolidated metrics table.

Each plot uses the row number in `history.csv` as the epoch number. The red star marks the first epoch where HR@10 reaches its best observed value.

Summary table rules:

- `epochs` equals the number of rows in `history.csv`.
- `best_val_hr_at_10` comes from the last row of `history.csv`.
- `best_val_ndcg_at_10` is the best observed `ndcg@10` value in `history.csv`.
- `test_hr_at_10` and `test_ndcg_at_10` come from `full_test_metrics.json` when that file exists.
- JSON and folder epoch values stay in separate audit columns.


In [29]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 130

CWD = Path.cwd()
if CWD.name == "experiments" and (CWD / "report.ipynb").exists():
    NOTEBOOK_DIR = CWD
elif (CWD / "main" / "experiments").exists():
    NOTEBOOK_DIR = CWD / "main" / "experiments"
else:
    NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else CWD

EXPERIMENTS_DIR = NOTEBOOK_DIR.resolve()
FIGURE_DIR = EXPERIMENTS_DIR
SUMMARY_CSV_PATH = EXPERIMENTS_DIR / "experiment_summary.csv"
SUMMARY_MARKDOWN_PATH = EXPERIMENTS_DIR / "experiment_summary.md"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment directory: {EXPERIMENTS_DIR}")
print(f"Figures and tables will be written to: {FIGURE_DIR}")

Experiment directory: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/main/experiments
Figures and tables will be written to: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/main/experiments


In [30]:
NEGATIVE_MODE_ALIASES = {
    "pn": "penalize-negative",
    "tap": "treat-as-positive",
    "ft": "embedding-finetune",
}


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def optional_float(value):
    if value is None:
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def parse_experiment_name(name: str) -> dict[str, object]:
    parts = name.split("_")
    parsed = {
        "folder_model": name,
        "folder_mode": "",
        "folder_mode_label": "",
        "folder_dataset": "",
        "folder_epochs": np.nan,
    }
    if len(parts) >= 4 and parts[-1].isdigit():
        parsed["folder_epochs"] = int(parts[-1])
        parsed["folder_dataset"] = parts[-2]
        parsed["folder_mode"] = parts[-3]
        parsed["folder_mode_label"] = NEGATIVE_MODE_ALIASES.get(parts[-3], parts[-3])
        parsed["folder_model"] = "_".join(parts[:-3])
    return parsed


def extract_identity(metrics: dict, parsed: dict[str, object]) -> dict[str, object]:
    config = metrics.get("config", {}) if isinstance(metrics, dict) else {}
    transfer = metrics.get("transfer", {}) if isinstance(metrics, dict) else {}
    dataset = (
        config.get("dataset")
        or config.get("target_dataset")
        or transfer.get("target_dataset")
        or parsed.get("folder_dataset")
    )
    model = config.get("model") or transfer.get("model") or parsed.get("folder_model")
    negative_items_handling = (
        config.get("negative_items_handling")
        or transfer.get("negative_items_handling")
        or parsed.get("folder_mode_label")
    )
    return {
        "dataset": dataset,
        "model": model,
        "negative_items_handling": negative_items_handling,
        "metrics_json_epochs": config.get("epochs", np.nan),
    }


def load_history(path: Path) -> pd.DataFrame:
    history = pd.read_csv(path)
    expected_columns = ["training_loss", "hr@10", "ndcg@10", "lr", "best_val_hr"]
    missing_columns = [column for column in expected_columns if column not in history.columns]
    if missing_columns:
        raise ValueError(f"{path} is missing required columns: {missing_columns}")
    for column in expected_columns:
        history[column] = pd.to_numeric(history[column], errors="coerce")
    history.insert(0, "epoch", np.arange(1, len(history) + 1))
    return history


def count_formatter(value, _position):
    if pd.isna(value):
        return ""
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if abs(value) >= 1_000:
        return f"{value / 1_000:.1f}K"
    return f"{value:.0f}"


def save_training_plot(experiment_dir: Path, history: pd.DataFrame) -> tuple[Path, dict[str, object]]:
    best_hr_idx = history["hr@10"].idxmax()
    best_ndcg_idx = history["ndcg@10"].idxmax()
    best_hr_epoch = int(history.loc[best_hr_idx, "epoch"])
    best_ndcg_epoch = int(history.loc[best_ndcg_idx, "epoch"])
    best_hr = float(history.loc[best_hr_idx, "hr@10"])
    best_ndcg = float(history.loc[best_ndcg_idx, "ndcg@10"])
    best_ndcg_at_best_hr = float(history.loc[best_hr_idx, "ndcg@10"])

    fig, ax_loss = plt.subplots(figsize=(12, 6))
    ax_metric = ax_loss.twinx()
    ax_lr = ax_loss.twinx()
    ax_lr.spines["right"].set_position(("axes", 1.12))

    loss_line = ax_loss.plot(
        history["epoch"],
        history["training_loss"],
        color="#4C78A8",
        linewidth=2,
        label="training loss",
    )
    hr_line = ax_metric.plot(
        history["epoch"],
        history["hr@10"],
        color="#E45756",
        linewidth=2,
        label="HR@10",
    )
    ndcg_line = ax_metric.plot(
        history["epoch"],
        history["ndcg@10"],
        color="#54A24B",
        linewidth=2,
        label="NDCG@10",
    )
    lr_line = ax_lr.plot(
        history["epoch"],
        history["lr"],
        color="#B279A2",
        linewidth=2,
        linestyle="--",
        label="learning rate",
    )

    metric_max = float(history[["hr@10", "ndcg@10"]].max(skipna=True).max())
    ax_metric.set_ylim(0, max(0.01, metric_max * 1.18))

    ax_loss.axvline(best_hr_epoch, color="#E45756", linestyle=":", linewidth=1.8)
    ax_metric.scatter(
        [best_hr_epoch],
        [best_hr],
        color="#E45756",
        marker="*",
        s=220,
        zorder=6,
    )
    ax_metric.annotate(
        f"best HR@10 {best_hr:.4f}\nepoch {best_hr_epoch}",
        xy=(best_hr_epoch, best_hr),
        xytext=(8, 14),
        textcoords="offset points",
        fontsize=9,
        color="#7A1F1F",
        bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#E45756", "alpha": 0.9},
    )

    ax_loss.set_xlabel("Epoch number")
    ax_loss.set_ylabel("Training loss")
    ax_metric.set_ylabel("Validation HR@10 / NDCG@10")
    ax_lr.set_ylabel("Learning rate")
    if history["lr"].dropna().gt(0).all():
        ax_lr.set_yscale("log")
    ax_loss.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    handles = loss_line + hr_line + ndcg_line + lr_line
    handles.append(
        Line2D([0], [0], color="#E45756", marker="*", linestyle="None", markersize=13, label="first best HR@10")
    )
    ax_loss.legend(handles=handles, loc="best")
    ax_loss.set_title(
        f"{experiment_dir.name}: training loss, ranking metrics, and learning rate",
        pad=16,
    )

    output_path = FIGURE_DIR / f"{experiment_dir.name}_training_curves.png"
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    return output_path, {
        "best_hr_epoch": best_hr_epoch,
        "best_hr_at_10": best_hr,
        "best_ndcg_epoch": best_ndcg_epoch,
        "best_ndcg_at_10": best_ndcg,
        "best_ndcg_at_best_hr_epoch": best_ndcg_at_best_hr,
    }


def summarize_experiment(experiment_dir: Path) -> dict[str, object]:
    parsed = parse_experiment_name(experiment_dir.name)
    metrics = read_json(experiment_dir / "metrics.json")
    full_test_metrics = read_json(experiment_dir / "full_test_metrics.json")
    identity = extract_identity(metrics, parsed)
    history = load_history(experiment_dir / "history.csv")
    figure_path, best_history = save_training_plot(experiment_dir, history)
    history_rows = int(history.shape[0])
    last_row = history.iloc[-1]
    has_full_test_metrics = bool(full_test_metrics)

    row = {
        "experiment": experiment_dir.name,
        **identity,
        **parsed,
        "epochs": history_rows,
        "history_rows": history_rows,
        "full_test_metrics_exists": has_full_test_metrics,
        "best_model_exists": (experiment_dir / "best_model.pt").exists(),
        "current_model_exists": (experiment_dir / "current_model.pt").exists(),
        "best_model_size_mb": round((experiment_dir / "best_model.pt").stat().st_size / (1024 * 1024), 2)
        if (experiment_dir / "best_model.pt").exists()
        else np.nan,
        "current_model_size_mb": round((experiment_dir / "current_model.pt").stat().st_size / (1024 * 1024), 2)
        if (experiment_dir / "current_model.pt").exists()
        else np.nan,
        **best_history,
        "final_training_loss": float(last_row["training_loss"]),
        "final_lr": float(last_row["lr"]),
        "final_val_hr_at_10": float(last_row["hr@10"]),
        "final_val_ndcg_at_10": float(last_row["ndcg@10"]),
        "best_val_hr_at_10": float(last_row["best_val_hr"]),
        "test_hr_at_10": optional_float(full_test_metrics.get("test_hr_at_10")),
        "test_ndcg_at_10": optional_float(full_test_metrics.get("test_ndcg_at_10")),
        "test_metrics_source": "full_test_metrics.json" if has_full_test_metrics else "",
        "metrics_json_best_val_hr_at_10": optional_float(metrics.get("best_val_hr_at_10")),
        "metrics_json_test_hr_at_10": optional_float(metrics.get("test_hr_at_10")),
        "metrics_json_test_ndcg_at_10": optional_float(metrics.get("test_ndcg_at_10")),
        "full_test_hr_at_10": optional_float(full_test_metrics.get("full_test_hr_at_10")),
        "full_test_ndcg_at_10": optional_float(full_test_metrics.get("full_test_ndcg_at_10")),
        "figure_path": str(figure_path.relative_to(EXPERIMENTS_DIR)),
    }
    return row


In [31]:
experiment_dirs = sorted(
    path for path in EXPERIMENTS_DIR.iterdir()
    if path.is_dir() and (path / "history.csv").exists()
)

if not experiment_dirs:
    raise FileNotFoundError(f"No experiment folders with history.csv found under {EXPERIMENTS_DIR}")

experiment_index_df = pd.DataFrame(
    [
        {
            "experiment": path.name,
            "has_history": (path / "history.csv").exists(),
            "has_metrics": (path / "metrics.json").exists(),
            "has_full_test_metrics": (path / "full_test_metrics.json").exists(),
            "has_best_model": (path / "best_model.pt").exists(),
            "has_current_model": (path / "current_model.pt").exists(),
        }
        for path in experiment_dirs
    ]
)
display(experiment_index_df)

,experiment,has_history,has_metrics,has_full_test_metrics,has_best_model,has_current_model
0,sasrec_pn_mobilerec_100,True,True,True,True,True
1,sasrec_tap_mobilerec_100,True,True,True,True,True
2,tisasrec_m_ft_steamrec_10,True,True,False,True,True
3,tisasrec_m_ft_steamrec_60,True,True,True,True,True
4,tisasrec_m_pn_mobilerec_100,True,True,True,True,True
5,tisasrec_m_pn_mobilerec_15,True,True,False,True,True
6,tisasrec_m_pn_steamrec_100,True,True,True,True,True
7,tisasrec_m_pn_steamrec_15,True,True,False,True,True


In [32]:
summary_rows = [summarize_experiment(experiment_dir) for experiment_dir in experiment_dirs]
experiment_summary_df = pd.DataFrame(summary_rows)

sort_columns = ["dataset", "model", "negative_items_handling", "epochs", "experiment"]
existing_sort_columns = [column for column in sort_columns if column in experiment_summary_df.columns]
experiment_summary_df = experiment_summary_df.sort_values(existing_sort_columns).reset_index(drop=True)

print("Generated figures:")
for figure_name in experiment_summary_df["figure_path"]:
    print(f"- {figure_name}")

ordered_columns = [
    "experiment",
    "dataset",
    "model",
    "negative_items_handling",
    "best_hr_at_10",
    "best_ndcg_at_10",
    "epochs",
    "best_hr_epoch",
    "best_ndcg_epoch",
    "full_test_hr_at_10",
    "full_test_ndcg_at_10",
]
ordered_columns = [column for column in ordered_columns if column in experiment_summary_df.columns]
experiment_summary_df = experiment_summary_df[ordered_columns]


experiment_summary_df = experiment_summary_df.sort_values(["best_hr_at_10", "best_ndcg_at_10"], ascending=[False, False]).reset_index(drop=True)

experiment_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
SUMMARY_MARKDOWN_PATH.write_text(experiment_summary_df.to_markdown(index=False), encoding="utf-8")

print(f"Wrote summary CSV: {SUMMARY_CSV_PATH}")
print(f"Wrote summary markdown: {SUMMARY_MARKDOWN_PATH}")

display(experiment_summary_df)

Generated figures:
- sasrec_pn_mobilerec_100_training_curves.png
- sasrec_tap_mobilerec_100_training_curves.png
- tisasrec_m_pn_mobilerec_15_training_curves.png
- tisasrec_m_pn_mobilerec_100_training_curves.png
- tisasrec_m_ft_steamrec_10_training_curves.png
- tisasrec_m_pn_steamrec_15_training_curves.png
- tisasrec_m_ft_steamrec_60_training_curves.png
- tisasrec_m_pn_steamrec_100_training_curves.png
Wrote summary CSV: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/main/experiments/experiment_summary.csv
Wrote summary markdown: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/main/experiments/experiment_summary.md


,experiment,dataset,model,negative_items_handling,best_hr_at_10,best_ndcg_at_10,epochs,best_hr_epoch,best_ndcg_epoch,full_test_hr_at_10,full_test_ndcg_at_10
0,tisasrec_m_pn_mobilerec_100,mobilerec,tisasrec_m,penalize-negative,0.218180,0.103847,69,24,24,0.004004,0.001834
1,tisasrec_m_pn_mobilerec_15,mobilerec,tisasrec_m,penalize-negative,0.187428,0.090002,15,15,15,NaN,NaN
2,tisasrec_m_ft_steamrec_10,steamrec,tisasrec_m,penalize-negative,0.142645,0.066148,10,7,8,NaN,NaN
3,tisasrec_m_ft_steamrec_60,steamrec,tisasrec_m,penalize-negative,0.142645,0.066148,60,7,8,0.000906,0.000387
4,sasrec_tap_mobilerec_100,mobilerec,sasrec,treat-as-positive,0.141463,0.085005,100,96,96,0.006630,0.003087
5,sasrec_pn_mobilerec_100,mobilerec,sasrec,penalize-negative,0.136736,0.065074,100,59,59,0.001583,0.000773
6,tisasrec_m_pn_steamrec_15,steamrec,tisasrec_m,penalize-negative,0.115808,0.054964,15,12,6,NaN,NaN
7,tisasrec_m_pn_steamrec_100,steamrec,tisasrec_m,penalize-negative,0.115808,0.054964,100,12,6,0.002025,0.000740
